# Python Decorators: Practice Problems

These problems go from basic to advanced and cover the main ways to write decorators (plain functions, decorators with arguments, class-based decorators, decorators that work with or without arguments, decorators applied to classes) and the most common real-world uses (logging, timing, caching, retries, access control, validation, registration, rate limiting).

**Suggested path:** do Problems 1–4 first, since everything else builds on them. Problems 9–12 are where most people get stuck (the "three layers of functions" idea), so take your time there.

Each problem has an empty code cell below it for your solution.

## Level 1: Fundamentals

### Problem 1: Hello decorator

Write a decorator `announce` that prints `"Calling function..."` before the decorated function runs and `"Done!"` after it finishes. Apply it to a function `greet(name)` that prints a greeting.

In [ ]:
# Problem 1: your solution here
def announce(func):
    def wrapper():
        print("Calling function...")
        func()
        print("Done!")
    return wrapper

@announce
def greet():
    print("Hello, World!")

greet()

### Problem 2: Handle any arguments and return values

Modify `announce` so it works on functions with any number of positional and keyword arguments, and so it passes along the decorated function's return value. Test it on `add(a, b)` and `power(base, exp=2)`.

In [34]:
# Problem 2: your solution here
from typing import Any, Callable


def announce(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args, **kwargs):
        print("Calling function...")
        result = func(*args, **kwargs)
        print(func.__name__, func.__doc__)
        print("Done!")
        return result

    return wrapper


@announce
def add(a: int, b: int) -> int:
    return a + b


@announce
def power(base: int, exp: int = 2) -> int:
    return base**exp


print(add(1, 2))
print(power(3, 4))

Calling function...
add None
Done!
3
Calling function...
power None
Done!
81


### Problem 3: Preserve metadata

After decorating `add` with your decorator, print `add.__name__` and `add.__doc__`. Explain why they show the wrong values, then fix the problem with `functools.wraps`.

In [ ]:
# Problem 3: your solution here
print(add.__name__, add.__doc__)

### Problem 4: Without the `@` syntax

Apply a decorator to a function *without* the `@` symbol, using plain function calls. Then explain in one or two sentences what `@decorator` actually does behind the scenes.

In [ ]:
# Problem 4: your solution here
d = announce(add) # This wraps the add function so now d points to `wrapper`
e = d(1,2) # Wrapper can accept any arguments. All the arguments will get passed to the add function. 
print(e)


## Level 2: Practical Decorators

### Problem 5: Timer

Write a decorator `timer` that measures how long a function takes and prints something like `slow_function took 1.0023 seconds`. Test it on a function that calls `time.sleep(1)`.

In [ ]:
# Problem 5: your solution here
import time


def timer(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: tuple, **kwargs: dict) -> Any:
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()

        print(f"Total time taken: {end_time - start_time}")
        return result

    return wrapper


@timer
def caller():
    time.sleep(1)

print(caller())

### Problem 6: Logger

Write a decorator `log_calls` that logs the function name, the arguments it received, and its return value. For example, `multiply(3, 4)` should print:

```
Calling multiply(3, 4)
multiply returned 12
```

In [ ]:
# Problem 6: your solution here
def log_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args, **kwargs):
        arguments = tuple(x for x in args)
        kwarguments = tuple({key: value} for key, value in kwargs)

        print(f"Calling {func.__name__}{arguments + kwarguments}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result

    return wrapper


@log_calls
def multiply(a: int, b: int, c: int = 3) -> int:
    return a * b * c


multiply(3, 4)

### Problem 7: Call counter (decorator with state)

Write a decorator `count_calls` that tracks how many times a function has been called. The count should be available as an attribute, so after calling `say_hi()` three times, `say_hi.calls` equals `3`.

In [ ]:
# Problem 7: your solution here
def count_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args, **kwargs):
        wrapper.count += 1
        return func(*args, **kwargs)

    # setattr(wrapper, "count", 0)
    wrapper.count = 0

    return wrapper


@count_calls
def say_hi():
    print("Hi!")


say_hi()
say_hi()
say_hi()

print(say_hi.count)

### Problem 8: Memoization

Write your own `memoize` decorator that caches results based on the arguments. Apply it to a recursive `fibonacci(n)` and compare the speed of `fibonacci(35)` with and without it. Then compare your version with Python's built-in `functools.lru_cache`.

In [33]:
# Problem 8: your solution here
from functools import lru_cache

def memoize(func: Callable[..., Any]) -> Callable[..., Any]:
    called: dict = {}

    def wrapper(*args, **kwargs):
        arguments = args + tuple(kwargs)

        if called.get(arguments) != None:
            return called[arguments]

        result = func(*args, **kwargs)
        called[arguments] = result
        return result

    return wrapper


@memoize
def fibonacci_1(n):
    if n < 2:
        return n
    return fibonacci_1(n - 1) + fibonacci_1(n - 2)

def fibonacci_2(n, k=1, prev=0):
    return k if n == 1 else fibonacci_2(n - 1, prev + k, k)

@lru_cache(maxsize=None)
def fibonacci_3(n):
    if n < 2:
        return n
    return fibonacci_3(n - 1) + fibonacci_3(n - 2)


for fun in [fibonacci_1, fibonacci_1, fibonacci_2, fibonacci_3]:
    start_time = time.time()
    print(fun(23))
    end_time = time.time()

    print(f"Time required to calculate({fun.__name__}): {end_time - start_time}")

28657
Time required to calculate(wrapper): 0.0002818107604980469
28657
Time required to calculate(wrapper): 1.430511474609375e-05
28657
Time required to calculate(fibonacci_2): 1.621246337890625e-05
28657
Time required to calculate(fibonacci_3): 3.266334533691406e-05


## Level 3: Decorators with Arguments

### Problem 9: Repeat N times

Write a decorator `repeat(n)` that runs the decorated function `n` times.

```python
@repeat(3)
def say(msg):
    print(msg)

say("Hi")  # prints "Hi" three times
```

In [35]:
# Problem 9: your solution here
def repeat(count: int = 1) -> Callable[..., Any]:
    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        def wrapper(*args, **kwargs):
            for _ in range(count):
                func(*args, **kwargs)
            return

        return wrapper

    return decorator

@repeat(3)
def say(msg):
    print(msg)

say("Hi")  # prints "Hi" three times

Hi
Hi
Hi


### Problem 10: Retry on failure

Write a decorator `retry(max_attempts=3, delay=1, exceptions=(Exception,))` that re-runs a function if it raises one of the specified exceptions, waiting `delay` seconds between attempts. If every attempt fails, re-raise the last exception. Test it with a function that randomly fails.

In [46]:
# Problem 10: your solution here
import random


def retry(max_attempts=1, delay=0, exceptions=(Exception,)) -> Callable[..., Any]:
    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        def wrapper(*args, **kwargs):
            for tries in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(e)
                    if tries < max_attempts - 1:
                        time.sleep(delay)

        return wrapper

    return decorator

@retry(max_attempts=3, delay=1)
def rain_guess():
    chance = random.random()

    if chance > 0.75:
        raise Exception(f"{chance * 100}% Chances of rain.")

    return "Unlikely to rain."

rain_guess()

85.69119016494865% Chances of rain.
78.14600919232613% Chances of rain.


'Unlikely to rain.'

### Problem 11: Type validation

Write a decorator `validate_types` that takes expected types for each argument and raises a `TypeError` if the call doesn't match.

```python
@validate_types(int, int)
def add(a, b):
    return a + b

add(2, 3)     # 5
add(2, "3")   # TypeError
```

*Bonus:* read the expected types from the function's type hints instead of passing them in.

In [48]:
# Problem 11: your solution here
def validate_types(*required_type: type) -> Callable[..., Any]:

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        def wrapper(*args, **kwargs) -> Any:
            nonlocal required_type
            combined_args = args + tuple(kwargs.values())

            if len(combined_args) != len(required_type):
                raise TypeError("Argument count mismatch.")

            for arg, req in zip(combined_args, required_type):
                if not isinstance(arg, req):
                    raise TypeError(f"Argument type of {req} expected but got {type(arg).__name__}.")
        
            return

        return wrapper

    return decorator

@validate_types(int, int)
def add(a, b):
    return a + b

add(2, 3)     # 5
add(2, "3")   # TypeError

TypeError: Argument type of <class 'int'> expected but got str.

### Problem 12: Optional arguments

Write a decorator `debug` that works both with and without parentheses:

```python
@debug
def f(): ...

@debug(prefix="[DEBUG]")
def g(): ...
```

In [ ]:
# Problem 12: your solution here

## Level 4: Class-Based and Advanced Forms

### Problem 13: Decorator as a class

Rewrite the `count_calls` decorator from Problem 7 as a class that uses `__init__` and `__call__`. Explain when a class-based decorator might be better than a function-based one.

In [ ]:
# Problem 13: your solution here

### Problem 14: Stacking decorators

Apply both `timer` and `log_calls` to the same function. Predict the output order before running it, then swap the order of the decorators and explain the difference.

In [ ]:
# Problem 14: your solution here

### Problem 15: Decorating a class

Write a class decorator `singleton` that ensures only one instance of a class is ever created. Every call to `Database()` should return the same object.

In [ ]:
# Problem 15: your solution here

### Problem 16: Add methods to a class

Write a class decorator `add_repr` that automatically adds a `__repr__` method displaying all instance attributes, e.g. `Point(x=1, y=2)`. Compare the result with what `@dataclass` gives you.

In [ ]:
# Problem 16: your solution here

## Level 5: Real-World Scenarios

### Problem 17: Access control

Suppose there's a global `current_user = {"name": "alice", "role": "viewer"}`. Write a decorator `requires_role("admin")` that raises a `PermissionError` if the current user doesn't have the required role. Apply it to a `delete_user()` function.

In [ ]:
# Problem 17: your solution here

### Problem 18: Function registry (plugin / routing pattern)

Build a simple command registry similar to how Flask registers routes:

```python
@command("hello")
def hello():
    print("Hello!")

@command("bye")
def bye():
    print("Goodbye!")

run("hello")  # Hello!
```

The decorator should store functions in a dictionary and return the original function unchanged.

In [ ]:
# Problem 18: your solution here

### Problem 19: Rate limiter

Write a decorator `rate_limit(calls, period)` that allows a function to run at most `calls` times per `period` seconds. Extra calls should raise an exception (or, as a variation, wait until they're allowed).

In [ ]:
# Problem 19: your solution here

### Problem 20: Deprecation warning

Write a decorator `deprecated(reason)` that issues a `DeprecationWarning` using the `warnings` module whenever the function is called, including the reason in the message.

In [ ]:
# Problem 20: your solution here

### Problem 21: Async decorator

Write a `timer` decorator that works on `async def` functions.

*Bonus:* make one decorator that works on both regular and async functions (hint: `inspect.iscoroutinefunction`).

*Note:* in Jupyter you can `await` a coroutine directly in a cell, without `asyncio.run()`.

In [ ]:
# Problem 21: your solution here

### Problem 22: Built-in decorators

Create a `Temperature` class that uses:

- `@property`, with a setter that rejects values below absolute zero
- `@classmethod`, for an alternate constructor `from_fahrenheit`
- `@staticmethod`, for a helper `celsius_to_kelvin`

Explain what each built-in decorator does.

In [ ]:
# Problem 22: your solution here